# Hyperparameter Tuning
- Using cross validation approach

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit


import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support


## Input

In [ ]:
# get path for folders
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
data_folder = os.path.join(project_root, "data")
results_folder = os.path.join(project_root, "results")

In [ ]:
# training data
df = pd.read_csv(f"{data_folder}/DroughtITS_mapping_w_labels_training_data.csv")

# Data processing
cat_features = ['plant']

# training data (data from 2022)
df_encoded = pd.get_dummies(df, columns=cat_features)
df_encoded = df_encoded.rename(columns={'plant_Rubber tree': 'plant_Rubber_tree'})

In [ ]:
with open(f"{results_folder}/group_names/pathogen_group_names_training_data.txt", "r") as file:
    group_names = [line.strip() for line in file.readlines()]

In [ ]:
# we don't run Fusarium spp. because it appears in all samples
group_names.remove("Fusarium spp.")

## Feature Sets

In [ ]:
features_sets = {'features_set1':  ['drought', 'water_content', 'organic_matter',
       'nitrogen', 'phosphorus', 'potassium', 'temp_soil', 'pH',
       'plant_Cassava', 'plant_Rice', 'plant_Rubber_tree', 'plant_Sugarcane'],
                 'features_set2': ['drought', 'water_content', 'organic_matter',
       'nitrogen', 'phosphorus', 'potassium', 'temp_soil', 'pH'],
                 'features_set3': ['drought', 'water_content', 'organic_matter',
       'nitrogen', 'phosphorus', 'potassium', 'pH'],
                 'features_set4': ['drought', 'water_content', 'organic_matter',
       'nitrogen', 'phosphorus', 'potassium', 'pH',
       'plant_Cassava', 'plant_Rice', 'plant_Rubber_tree', 'plant_Sugarcane']}

## Cross validation parameters

In [ ]:
cv_plan = {
    "Acremonium spp.": {
        "method": "StratifiedKFold",
        "k": 3,
        "reason": "Minority class size m=9; 3-fold CV ensures ~3 minority samples per fold for stable hyperparameter tuning."
    },
    "Candida tropicalis": {
        "method": "StratifiedKFold",
        "k": 3,
        "reason": "Minority class size m=9; 3-fold CV provides sufficient minority representation per fold."
    },
    "Curvularia lunata": {
        "method": "fixed",
        "reason": "Minority class size m=3 is too small for reliable cross-validation; hyperparameters are fixed to conservative values."
    },
    "Falciformispora senegalensis": {
        "method": "StratifiedKFold",
        "k": 3,
        "reason": "Minority class size m=9; 3-fold CV maintains at least ~3 minority samples per fold."
    },
    "Lichtheimia spp.": {
        "method": "fixed",
        "reason": "Minority class size m=3 is insufficient for stable cross-validation; conservative fixed hyperparameters are used."
    },
    "Mucor spp.": {
        "method": "StratifiedKFold",
        "k": 2,
        "reason": "Minority class size m=4; 2-fold CV ensures at least two minority samples per validation fold."
    },
    "Rhizopus spp.": {
        "method": "StratifiedKFold",
        "k": 5,
        "reason": "Minority class size m=32; 5-fold CV provides stable and well-balanced validation splits."
    },
    "Scedosporium spp.": {
        "method": "StratifiedKFold",
        "k": 5,
        "reason": "Minority class size m=59; sufficient samples to support standard 5-fold stratified CV."
    },
    "Talaromyces marneffei": {
        "method": "StratifiedKFold",
        "k": 5,
        "reason": "Minority class size m=34; adequate for 5-fold stratified cross-validation."
    }
}


## Hyper parameters tuning

### Helpers

In [ ]:
# -----------------------------
# 1) Rule-based "absence prediction" from a fitted tree
#    Rule = leaf where ALL training samples are absence (y==0)
#           AND number of absence samples in that leaf >= min_absence_in_leaf
#    Prediction: if sample lands in a rule-leaf => predict 0, else predict 1
# -----------------------------
def _get_rule_leaves_absence_only(tree_clf, X_train, y_train, min_absence_in_leaf=2):
    """
    Returns a set of leaf node IDs that qualify as "absence rules"
    based on training data only.
    """
    y_train = np.asarray(y_train)
    leaf_ids = tree_clf.apply(X_train)

    rule_leaves = set()
    for leaf in np.unique(leaf_ids):
        idx = (leaf_ids == leaf)
        y_leaf = y_train[idx]
        # pure absence leaf + minimum support in absence
        if np.all(y_leaf == 0) and np.sum(y_leaf == 0) >= min_absence_in_leaf:
            rule_leaves.add(int(leaf))
    return rule_leaves


def predict_absence_by_rules(tree_clf, X, rule_leaves):
    """
    Predict 0 if sample lands in a rule leaf, else 1.
    """
    leaf_ids = tree_clf.apply(X)
    preds = np.where(np.isin(leaf_ids, list(rule_leaves)), 0, 1)
    return preds


# -----------------------------
# 2) Evaluate one parameter setting with CV
# -----------------------------
def cv_score_rule_tree(
    X, y,
    *,
    max_depth,
    min_samples_leaf,
    cv_splits=5,
    random_state=42,
    min_absence_in_leaf=2
):
    """
    Returns mean metrics across folds:
      - precision0, recall0, f1_0 for absence class (class 0)
      - coverage0: fraction predicted as absence (rule fires)
      - rule_found_rate: fraction of folds where at least one rule leaf exists
    """
    X = np.asarray(X)
    y = np.asarray(y)

    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    fold_rows = []
    for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_va, y_va = X[va], y[va]

        clf = DecisionTreeClassifier(
            random_state=random_state,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf
        )
        clf.fit(X_tr, y_tr)

        rule_leaves = _get_rule_leaves_absence_only(
            clf, X_tr, y_tr,
            min_absence_in_leaf=min_absence_in_leaf
        )
        y_pred = predict_absence_by_rules(clf, X_va, rule_leaves)

        # Metrics for absence class (label 0)
        p, r, f1, _ = precision_recall_fscore_support(
            y_va, y_pred, labels=[0], average=None, zero_division=0
        )
        precision0 = float(p[0])
        recall0 = float(r[0])
        f1_0 = float(f1[0])

        coverage0 = float(np.mean(y_pred == 0))  # how often rule fires on validation
        rule_found = int(len(rule_leaves) > 0)

        fold_rows.append({
            "fold": fold,
            "precision0": precision0,
            "recall0": recall0,
            "f1_0": f1_0,
            "coverage0": coverage0,
            "rule_found": rule_found,
            "n_rules": len(rule_leaves),
        })

    df = pd.DataFrame(fold_rows)

    out = {
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "f1_0_mean": df["f1_0"].mean(),
        "precision0_mean": df["precision0"].mean(),
        "recall0_mean": df["recall0"].mean(),
        "coverage0_mean": df["coverage0"].mean(),
        "rule_found_rate": df["rule_found"].mean(),
        "avg_rules_per_fold": df["n_rules"].mean(),
        "fold_details": df
    }
    return out


# -----------------------------
# 3) Grid search over (max_depth, min_samples_leaf)
#    Selection rule:
#      1) highest mean F1 for class 0
#      2) tie-break: higher rule_found_rate
#      3) tie-break: higher coverage0_mean
#      4) tie-break: simpler tree (smaller max_depth; treat None as large)
# -----------------------------
def tune_rule_tree_params(
    X, y,
    *,
    max_depth_grid=(2, 3, 4, 5),
    min_samples_leaf_grid=(1, 2, 3, 5),
    cv_splits=5,
    random_state=42,
    min_absence_in_leaf=2
):
    results = []
    for md in max_depth_grid:
        for msl in min_samples_leaf_grid:
            res = cv_score_rule_tree(
                X, y,
                max_depth=md,
                min_samples_leaf=msl,
                cv_splits=cv_splits,
                random_state=random_state,
                min_absence_in_leaf=min_absence_in_leaf
            )
            results.append({
                "max_depth": md,
                "min_samples_leaf": msl,
                "f1_0_mean": res["f1_0_mean"],
                "precision0_mean": res["precision0_mean"],
                "recall0_mean": res["recall0_mean"],
                "coverage0_mean": res["coverage0_mean"],
                "rule_found_rate": res["rule_found_rate"],
                "avg_rules_per_fold": res["avg_rules_per_fold"],
                "_details": res["fold_details"],  # keep for inspection
            })

    df = pd.DataFrame(results)

    def depth_sort_value(d):
        return 10_000 if d is None else int(d)

    df["_depth_sort"] = df["max_depth"].map(depth_sort_value)

    df_sorted = df.sort_values(
        by=["f1_0_mean", "rule_found_rate", "coverage0_mean", "_depth_sort", "min_samples_leaf"],
        ascending=[False, False, False, True, True],
        kind="mergesort"
    ).drop(columns=["_depth_sort"])

    best = df_sorted.iloc[0].to_dict()
    return df_sorted, best


### Find best parameter set

In [ ]:
# Loop over all group_names, but only run tuning for groups where cv_plan[method] == "StratifiedKFold"
# For each such group, tune hyperparameters for all feature_sets, then save results

for group_name in group_names:
    if cv_plan[group_name]["method"] == "fixed":
        print(f"=== Using fixed params for group: {group_name} ===")
        
        best_params_per_set = {}
        
        for set_name in features_sets.keys():
            best_params_per_set[set_name] = {
                "max_depth": 3,
                "min_samples_leaf": 2,
                # Add other keys to match structure, but since fixed, minimal
            }
                
        save_path = f'{results_folder}/cross_validation/best_params/{group_name}/best_params_per_set.json'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        with open(save_path, 'w') as f:
            json.dump(best_params_per_set, f)
        
        print(f"Saved fixed best_params_per_set to {save_path}")
        continue
    
    plan = cv_plan[group_name]
    print(f"=== Tuning for group: {group_name} ===")
    
    y = df_encoded[group_name]
    
    best_params_per_set = {}
    
    for set_name, feature_list in features_sets.items():
        print(f"Tuning for feature set: {set_name}")
        
        X = df_encoded[feature_list].values
        
        df_grid, best = tune_rule_tree_params(
            X, y,
            max_depth_grid=(2, 3, 4),
            min_samples_leaf_grid=(1, 2, 3),
            cv_splits=plan["k"],
            random_state=42
        )
        
        best_params_per_set[set_name] = best
        
        print(f"Best params: { {k: best[k] for k in ['max_depth', 'min_samples_leaf']} }")
    
    # Save results
    save_path = f'{results_folder}/cross_validation/best_params/{group_name}/best_params_per_set.json'
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    for set_name in best_params_per_set:
        details_df = best_params_per_set[set_name].pop("_details")
        details_path = f'{results_folder}/cross_validation/best_params/{group_name}/details_{set_name}.csv'
        details_df.to_csv(details_path, index=False)
        print(f"Saved details for {set_name} to {details_path}")
    
    with open(save_path, 'w') as f:
        json.dump(best_params_per_set, f)
    
    print(f"Saved best_params_per_set to {save_path}")

### Summarize best parameter set(s) for each group

In [ ]:
# Assuming the tuning loop has been run and best_params_per_set is available for each group
# If not, you can load from the saved JSON files

# List to collect summary rows
summary_rows = []

for group_name in group_names:
    save_path = f'{results_folder}/cross_validation/best_params/{group_name}/best_params_per_set.json'
    if os.path.exists(save_path):
        with open(save_path, 'r') as f:
            best_params_per_set = json.load(f)
        
        # Get CV info from cv_plan
        cv_info = cv_plan.get(group_name, {})
        cv_folds = cv_info.get("k", "fixed")  # Use k if available, else "fixed"
        
        for set_name, params in best_params_per_set.items():
            # Map set_name to display name, e.g., 'features_set1' -> 'FS1'
            display_set_name = 'FS' + set_name[-1] if set_name.startswith('features_set') else set_name
            row = {
                'Group': group_name,
                'Feature Set': display_set_name,
                'CV Folds': cv_folds,
                'Max Depth': params.get('max_depth'),
                'Min Samples Leaf': params.get('min_samples_leaf'),
                'F1_0 Mean': params.get('f1_0_mean'),
                'Precision0 Mean': params.get('precision0_mean'),
                'Recall0 Mean': params.get('recall0_mean'),
                'Coverage0 Mean': params.get('coverage0_mean'),
                'Rule Found Rate': params.get('rule_found_rate'),
                'Avg Rules Per Fold': params.get('avg_rules_per_fold')
            }
            summary_rows.append(row)

# Create DataFrame and display as table
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_excel(f'{results_folder}/cross_validation/best_params/summary_of_best_params.xlsx', index=False)